# Lab4-Assignment about Named Entity Recognition and Classification

This notebook describes the assignment of Lab 4 of the text mining course. We assume you have succesfully completed Lab1, Lab2 and Lab3 as welll. Especially Lab2 is important for completing this assignment.

**Learning goals**
* going from linguistic input format to representing it in a feature space
* working with pretrained word embeddings
* train a supervised classifier (SVM)
* evaluate a supervised classifier (SVM)
* learn how to interpret the system output and the evaluation results
* be able to propose future improvements based on the observed results


## Credits
This notebook was originally created by Marten Postma and Filip Ilievski and adapted by Piek vossen

## [Points: 18] Exercise 1 (NERC): Training and evaluating an SVM using CoNLL-2003

**[4 point] a) Load the CoNLL-2003 training data using the *ConllCorpusReader* and create for both *train.txt* and *test.txt*:**

    [2 points]  -a list of dictionaries representing the features for each training instances, e..g,
    ```
    [
    {'words': 'EU', 'pos': 'NNP'}, 
    {'words': 'rejects', 'pos': 'VBZ'},
    ...
    ]
    ```

    [2 points] -the NERC labels associated with each training instance, e.g.,
    dictionaries, e.g.,
    ```
    [
    'B-ORG', 
    'O',
    ....
    ]
    ```

For the following code to work download the data set from [this link](https://drive.google.com/drive/folders/1LChp70lMFNL9BtNi0nVy-V-fCRkEobgr)

Make sure to store the download within this /lab4 folder.

In [ ]:
from nltk.corpus.reader import ConllCorpusReader
from pathlib import Path
### Adapt the path to point to the CONLL2003 folder on your local machine
cwd = Path.cwd()
conll_dir = cwd.joinpath('nerc_datasets/CONLL2003')

# train = ConllCorpusReader('/Users/piek/Desktop/ONDERWIJS/data/nerc_datasets/CONLL2003', 'train.txt', ['words', 'pos', 'ignore', 'chunk'])
train = ConllCorpusReader(str(conll_dir), 'train.txt', ['words', 'pos', 'ignore', 'chunk'])
training_features = []
training_gold_labels = []
for token, pos, ne_label in train.iob_words():
    a_dict = {
       'words' : token,
       'pos' : pos
    }
    training_features.append(a_dict)
    training_gold_labels.append(ne_label)
    
print(f"Sample feature: {training_features[0]}")
print(f"Sample label: {training_gold_labels[0]}")
   

In [ ]:
### Adapt the path to point to the CONLL2003 folder on your local machine
train = ConllCorpusReader(str(conll_dir), 'test.txt', ['words', 'pos', 'ignore', 'chunk'])

test_features = []
test_gold_labels = []
for token, pos, ne_label in train.iob_words():
    a_dict = {
       'words' : token,
       'pos' : pos
    }
    test_features.append(a_dict)
    test_gold_labels.append(ne_label)

print(f"Sample feature: {test_features[0]}")
print(f"Sample label: {test_gold_labels[0]}")

**[2 points] b) provide descriptive statistics about the training and test data:**
* How many instances are in train and test?
* Provide a frequency distribution of the NERC labels, i.e., how many times does each NERC label occur?
* Discuss to what extent the training and test data is balanced (equal amount of instances for each NERC label) and to what extent the training and test data differ?

Tip: you can use the following `Counter` functionality to generate frequency list of a list:

In [ ]:
from collections import Counter 

training_NERC_label_dist = Counter(training_gold_labels)
test_NERC_label_dist = Counter(test_gold_labels)

print(test_NERC_label_dist.items())

train_total = training_NERC_label_dist.total()
test_total = test_NERC_label_dist.total()

print(f"Training size: {train_total}")
print(f"Test size: {test_total}")


print("Training NERC label distribution:", training_NERC_label_dist)
print("Test NERC label distribution:", test_NERC_label_dist)


# Create the Table Header
print(f"\n{'Label':<10} | {'Train Ratio (%)':<15} | {'Test Ratio (%)':<15} | {'Diff (%)':<10}")
print("-" * 60)

for label, train_count in training_NERC_label_dist.items():
    train_ratio = (train_count / train_total) * 100
    
    # Safely get the count from test set (defaults to 0 if label is missing)
    test_count = test_NERC_label_dist.get(label, 0)
    test_ratio = (test_count / test_total) * 100
    
    diff = abs(train_ratio - test_ratio)
    
    print(f"{label:<10} | {train_ratio:>14.3f}% | {test_ratio:>14.3f}% | {diff:>9.3f}%")


Analysis Q1b

Both training and test datasets are heavily imbalanced towards label 'O' (outside). It roughly accounts for 83% of the labels in both data sets. In a truly balanced data set each of the 9 labels should have similar frequencies which would mean about 11% per label.

Comparing between the data sets we can find that most of label frequency rankings are the same however B-ORG and B-PER are flipped. Comparing the distribution of labels between sets, we can see that each label has roughly the the same amount of representaion in both sets with <1% difference between any given label.

**[2 points] c) Concatenate the train and test features (the list of dictionaries) into one list. Load it using the *DictVectorizer*. Afterwards, split it back to training and test.**

Tip: You’ve concatenated train and test into one list and then you’ve applied the DictVectorizer.
The order of the rows is maintained. You can hence use an index (number of training instances) to split the_array back into train and test. Do NOT use: `
from sklearn.model_selection import train_test_split` here.


In [ ]:
from sklearn.feature_extraction import DictVectorizer

In [ ]:
vec = DictVectorizer()
the_array = training_features + test_features
X = vec.fit_transform(the_array)

X_train = X[:len(training_features)]
X_test = X[len(training_features):]

**[4 points] d) Train the SVM using the train features and labels and evaluate on the test data. Provide a classification report (sklearn.metrics.classification_report).**
The train (*lin_clf.fit*) might take a while. On my computer, it took 1min 53s, which is acceptable. Training models normally takes much longer. If it takes more than 5 minutes, you can use a subset for training. Describe the results:
* Which NERC labels does the classifier perform well on? Why do you think this is the case?
* Which NERC labels does the classifier perform poorly on? Why do you think this is the case?

In [ ]:
from sklearn import svm

In [ ]:
lin_clf = svm.LinearSVC()

In [ ]:
##### [ YOUR CODE SHOULD GO HERE ]
from sklearn.metrics import classification_report

# Create linear classifier
lin_clf.fit(X_train,training_gold_labels)

In [ ]:
# Predict and evaluate classifier
predictions = lin_clf.predict(X_test)
print(classification_report(test_gold_labels,predictions))

Analysis Q1d

Well performing labels: O, B-LOC
The top performing label is 'O' with 99% percision with a distant second of B-LOC having 81%.

The performance of theses labels match their higher frequency in the label frequency distribution of the training data. This implies that having a massive amount of data to learn a label from increases performance.

B-LOC appears to perform much better than labels with similar distrbution possibly because locations follow a stricter pattern of occurence in text. Locations usually appear after prepositions such as "at" or "in" making it easier to guess.

Poorly performing labels: I-PER, I-ORG, I-LOC, B-PER,

These labels performed terribly on the test set and this seems to reflect the lack of availability of these labels in the data set.

I-PER is the worst of the labels. The model appears to be randomly guessing (precision 33%) but able to recall specific names from the training data (recall 87%)

I-ORG and I-LOC suffer similar fates since they have fewer instances. The model has less to learn from and it difficult to learn the name of an org/loc that is long an varied compared to just recoginizing the starting of them (B-types). This is due to similar reasons of patterned based appearance described for B-LOC

Since I-type labels are inheriently rarer (since they must follow a B-type) the model struggles heavily with class imbalance

B-PER has high percision (87%) but low recall (44%) suggesting that the model is very conservative in it's predictions. Only if it is sure it is a name will it predice as such, however it has missed many valid names. 



**[6 points] e) Train a model that uses the embeddings of these words as inputs. Test again on the same data as in 2d. Generate a classification report and compare the results with the classifier you built in 2d.**

In [ ]:
# your code here
from gensim.models import Word2Vec
from sklearn import svm
import numpy as np
from sklearn.metrics import classification_report

  

In [ ]:
train_reader = ConllCorpusReader(str(conll_dir), 'train.txt', ['words', 'pos', 'ignore', 'chunk'])
test_reader = ConllCorpusReader(str(conll_dir), 'test.txt', ['words', 'pos', 'ignore', 'chunk'])

train_sentences = [
    [word.lower() for word, pos, label in sent]
    for sent in train_reader.iob_sents()
]

embedding_dim = 100

w2v_model = Word2Vec(
    sentences=train_sentences,
    vector_size=embedding_dim,
    window=5,
    min_count=1,
    workers=4,
    seed=42
)

X_train_embeddings = np.array([
    w2v_model.wv[feature["words"].lower()]
    for feature in training_features
])

X_test_embeddings = np.array([
    w2v_model.wv[feature["words"].lower()]
    if feature["words"].lower() in w2v_model.wv
    else np.zeros(embedding_dim)
    for feature in test_features
])


X_train_subset = X_train_embeddings[:25000]
y_train_subset = training_gold_labels[:25000]

In [ ]:
embed_clf = svm.LinearSVC(
    max_iter=2000,
    dual=False,
    class_weight="balanced"
)

embed_clf.fit(X_train_subset, y_train_subset)

embed_predictions = embed_clf.predict(X_test_embeddings)

print(classification_report(test_gold_labels, embed_predictions))

Analysis Q1e

A subset of 25000 was used since running it with the full set was taking over 20 minutes

O is once again the leading label with 96% precision and B-LOC is second again with a precision of 43%

B-LOC performed significantly worse compared to its perfomance in Q1d, this may have something to do with using embedding but could also be because of using a subset

Poorly performing labels: B-MISC, B-ORG, B-PER, I-LOC, I-MISC, I-ORG, I-PER

Most labels seemed to do poorly this time, with the only one holding up being the O label.

Although the B labels tended to do much better then the I labels in Q1d, in this question they seem to have done about the same, with I-Org actually outperforming most of the other labels.

Overall, since the best performing label didn't see too much of a dip in quality, the weighted averages didn't tank as much as they otherwise would've, for example with precision only going down 11% to 83% despite most of the individual precision values being under 15%.

Embeddings seem able to capture some general pattern, but are much weaker at specific distinctions

## [Points: 10] Exercise 2 (NERC): feature inspection using the [Annotated Corpus for Named Entity Recognition](https://www.kaggle.com/abhinavwalia95/entity-annotated-corpus)
**[6 points] a. Perform the same steps as in the previous exercise. Make sure you end up for both the training part (*df_train*) and the test part (*df_test*) with:**
* the features representation using **DictVectorizer**
* the NERC labels in a list

Please note that this is the same setup as in the previous exercise:
* load both train and test using:
    * list of dictionaries for features
    * list of NERC labels
* combine train and test features in a list and represent them using one hot encoding
* train using the training features and NERC labels

In [ ]:
import pandas
from pathlib import Path

In [ ]:
##### Adapt the path to point to your local copy of NERC_datasets (see Lab4a.1-NERC-introduction.ipynb)
cwd = Path.cwd()
ner_v2_dir = cwd.joinpath('ner_dataset.csv')
# path = '/Users/piek/Desktop/ONDERWIJS/data/nerc_datasets/kaggle/ner_v2.csv'
kaggle_dataset = pandas.read_csv(ner_v2_dir, on_bad_lines='skip', encoding='unicode_escape')

In [ ]:
len(kaggle_dataset)

In [ ]:
df_train = kaggle_dataset[:100000]
df_test = kaggle_dataset[100000:120000]
print(len(df_train), len(df_test))

**Load list of dictionaries for features and list of NERC labels**

In [ ]:
df_train.columns

In [ ]:
training_features = [{'words': word, 'pos': pos} for word, pos in zip(df_train['Word'], df_train['POS'])]
training_nerc_labels = df_train['Tag'].tolist()

print(f"Sample feature: {training_features[0]}")
print(f"Sample label: {training_nerc_labels[0]}")

In [ ]:
test_features = [{'words': word, 'pos': pos} for word, pos in zip(df_test['Word'], df_test['POS'])]
test_nerc_labels = df_test['Tag'].tolist()

print(f"Sample feature: {test_features[1]}")
print(f"Sample label: {test_nerc_labels[1]}")

**Provide descriptive statistics about the training and test data:**
* How many instances are in train and test?
* Provide a frequency distribution of the NERC labels, i.e., how many times does each NERC label occur?
* Discuss to what extent the training and test data is balanced (equal amount of instances for each NERC label) and to what extent the training and test data differ?

Tip: you can use the following `Counter` functionality to generate frequency list of a list:

In [ ]:
from collections import Counter 

training_nerc_label_dist = Counter(training_nerc_labels)
test_nerc_label_dist = Counter(test_nerc_labels)

train_total = training_nerc_label_dist.total()
test_total = test_nerc_label_dist.total()

print(f"Training size: {train_total}")
print(f"Test size: {test_total}")

print("\nTraining NERC label distribution:", training_nerc_label_dist)
print("Test NERC label distribution:", test_nerc_label_dist)

# Create comparison table
print(f"\n{'Label':<10} | {'Train Ratio (%)':<15} | {'Test Ratio (%)':<15} | {'Diff (%)':<10}")
print("-" * 60)

for label in set(training_nerc_label_dist.keys()) | set(test_nerc_label_dist.keys()):
    train_count = training_nerc_label_dist.get(label, 0)
    test_count = test_nerc_label_dist.get(label, 0)
    
    train_ratio = (train_count / train_total) * 100 if train_total > 0 else 0
    test_ratio = (test_count / test_total) * 100 if test_total > 0 else 0
    
    diff = abs(train_ratio - test_ratio)
    
    print(f"{label:<10} | {train_ratio:>14.3f}% | {test_ratio:>14.3f}% | {diff:>9.3f}%")

Both datasets are imbalanced toward the 'O' label. It accounts for about 84% of the data in both datasets. In an ideal dataset all labels would be evenly distributed. Outside of that the distributions of every label in both the training and test sets are very similar. Another downside is that certain label classes are extremely rare. For exammple, 'B-eve' makes up only 0.053% of the train and 0% of the testing data.

**Concatenate the train and test features (the list of dictionaries) into one list. Load it using the *DictVectorizer*. Afterwards, split it back to training and test.**

Tip: You’ve concatenated train and test into one list and then you’ve applied the DictVectorizer.
The order of the rows is maintained. You can hence use an index (number of training instances) to split the_array back into train and test. Do NOT use: `
from sklearn.model_selection import train_test_split` here.

In [ ]:
from sklearn.feature_extraction import DictVectorizer

# Concatenate train and test features
all_features = training_features + test_features

# Apply DictVectorizer
vec = DictVectorizer()
X = vec.fit_transform(all_features)

# Split back into train and test
X_train = X[:len(training_features)]
X_test = X[len(training_features):]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

**[4 points] b. Train and evaluate the model and provide the classification report:**
* use the SVM to predict NERC labels on the test data
* evaluate the performance of the SVM on the test data

Analyze the performance per NERC label.

In [ ]:
from sklearn import svm
from sklearn.metrics import classification_report

# Create and train the LinearSVC classifier
lin_clf = svm.LinearSVC(max_iter=2000)
lin_clf.fit(X_train, training_nerc_labels)

# Make predictions on test data
predictions = lin_clf.predict(X_test)

# Generate classification report
print(classification_report(test_nerc_labels, predictions))

**Performance Per Label**
'O': The model is very accurate at predicting this label. It achieves an F1 score of .99 however this label made up the vast majority of the data so this is to be expected.

'B-gpe': The model performs well on this label achieving a F1 score of 0.94. This is most likely because geopolitical entities follow similar patterns

'B-tim': The model performs well on this label achieving a F1 score of 0.83. This is probably becaue this label had a reasonable training frequency.

'B-geo': The model performs well on this label achieving a F1 score of 0.78. This is also due to the label's reasonable training frequency.

'B-per': The model achieves a high precision (0.81) but low recal (0.53) on this label showing is had conservative predictions with it.

'B-nat': The model achieves a high precision (1.00) but low recal (0.50) on this label showing is had conservative predictions with it.

'B-art': The model achieves 0s across the board due to the labels low frequency in data.

'B-eve': The model achieves 0s across the board due to the labels low frequency in data.

'B-org': The model achieves mediocre scores across the board. This is probably due to the ambiguity behind organization names.

'I-nat': The model performs very well on this label achieving a F1 score of 0.89. 

'I-art': The model achieves 0s across the board due to the labels low frequency in data.

'I-gpe': The model achieves a high precision (1.00) but low recal (0.50) on this label showing is had conservative predictions with it.

'I-eve': The model achieves 0s across the board due to the labels low frequency in data.

'I-tim': The model performs terribly with this label (0.14 F1 score) despite it having a descent amount of sample data for this label.

'I-per': The model acheives high recall (0.90) with low precision (0.42) showing it is overpredicting this label.

'I-org': The model achieves mediocre scores across the board. This is probably due to the ambiguity behind organization names.

'I-geo': The model achieves mediocre scores across the board. This is much worse than the label's B-type counterpart due to class imbalance.


## End of this notebook